In [1]:
#Evaluate bilingual search performance

In [2]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
import random
import matplotlib.pyplot as plt

In [7]:
data_folder = "D:/Harshita Ajmani/Code_harshu/NLP/data/"
model = "intfloat/multilingual-e5-base"
num_query = 20  
random.seed(42)

In [ ]:
# Loading and Building Index

embeddings_en = np.load(f"{data_folder}/embeddings_en.npy").astype('float32')
embeddings_fr = np.load(f"{data_folder}/embeddings_fr.npy").astype('float32')

dataframe = pd.read_csv(f"{data_folder}/index_data.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer(model, device=device)

# Build FAISS indexes
dim      = embeddings_en.shape[1]
index_en = faiss.IndexFlatIP(dim)
index_fr = faiss.IndexFlatIP(dim)
index_en.add(embeddings_en)
index_fr.add(embeddings_fr)
print(f"{len(dataframe):,} records indexed")

#print(dim)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


46,468 records indexed
768


In [ ]:
#Auto-generate test queries

def generate_queries(lang, n):
    """
    Sample n records that have keywords. Use first 3 keywords as the query.
    The correct answer is that exact record.
    """
    # Filter records with keywords
    col = f"keywords_{lang}"
    valid = dataframe[dataframe[col].notna() & (dataframe[col] != '')].copy()
    sampled = valid.sample(n=n, random_state=42)

    queries = []
    for idx, row in sampled.iterrows():
        keywords = row[col].split(',')[:3]  # take first 3 keywords
        query    = " ".join(k.strip() for k in keywords)
        queries.append({
            "query":        query,
            "lang":         lang,
            "true_idx":     idx,           
            "true_title":   row[f"title_{lang}"],
        })
    return queries

en_queries = generate_queries("en", n=10)
fr_queries = generate_queries("fr", n=10)
all_queries = en_queries + fr_queries

print(f"Generated {len(all_queries)} test queries")
print(f"\nSample EN query: '{en_queries[0]['query']}'")
print(f"Expected title  : '{en_queries[0]['true_title']}'")
print(f"\nSample FR query: '{fr_queries[0]['query']}'")
print(f"Expected title  : '{fr_queries[0]['true_title']}'")



Generated 20 test queries

Sample EN query: 'digital economy and society digital technology and internet use digital technology and internet use by individuals and households'
Expected title  : 'Barriers to accessibility encountered by persons with disabilities or long-term conditions when watching or listening to online content, aged 15 years and over, Canada, 2024'

Sample FR query: 'Avenue Boulevard Camion'
Expected title  : 'Réseau routier'


In [9]:
#search function

def search(query, lang, top_k=5):
    query_vector = model.encode(
        f"query: {query}",
        normalize_embeddings=True
    ).astype('float32').reshape(1, -1)

    index = index_en if lang == "en" else index_fr
    scores, indices = index.search(query_vector, top_k)
    return indices[0].tolist(), scores[0].tolist()

In [7]:
#Evaluation

results = []
for q in all_queries:
    top_indices, top_scores = search(q["query"], q["lang"], top_k=5)
    hit = q["true_idx"] in top_indices

    results.append({
        "query":       q["query"],
        "lang":        q["lang"],
        "true_idx":    q["true_idx"],
        "true_title":  q["true_title"],
        "hit":         hit,
        "top_score":   round(top_scores[0], 4),
        "rank":        top_indices.index(q["true_idx"]) + 1 if hit else None,
    })

In [ ]:
results_dataframe = pd.DataFrame(results)

en_results = results_dataframe[results_dataframe["lang"] == "en"]
fr_results = results_dataframe[results_dataframe["lang"] == "fr"]

en_hit_rate = en_results["hit"].mean() * 100
fr_hit_rate = fr_results["hit"].mean() * 100
overall     = results_dataframe["hit"].mean() * 100

print(f"\n{'='*50}")
print(f"EVALUATION RESULTS")
print(f"{'='*50}")
print(f"  English Hit Rate @5  : {en_hit_rate:.1f}%  ({en_results['hit'].sum()}/10)")
print(f"  French  Hit Rate @5  : {fr_hit_rate:.1f}%  ({fr_results['hit'].sum()}/10)")
print(f"  Overall Hit Rate @5  : {overall:.1f}%  ({results_dataframe['hit'].sum()}/20)")
print(f"{'='*50}")

print(f"\nDetailed Results:")
for _, r in results_dataframe.iterrows():
    status = "Right" if r["hit"] else "Wrong"
    rank   = f"rank #{int(r['rank'])}" if r["hit"] else "not found"
    print(f"  {status} [{r['lang'].upper()}] '{r['query'][:50]}' → {rank} | score: {r['top_score']}")


EVALUATION RESULTS
  English Hit Rate @5  : 20.0%  (2/10)
  French  Hit Rate @5  : 50.0%  (5/10)
  Overall Hit Rate @5  : 35.0%  (7/20)

Detailed Results:
  Wrong [EN] 'digital economy and society digital technology and' → not found | score: 0.8894
  Wrong [EN] 'table travel and tourism travel and traveller char' → not found | score: 0.8587
  Right [EN] 'Software updates Device security Patch management' → rank #1 | score: 0.8819
  Wrong [EN] 'manufacturing table wood paper and printing' → not found | score: 0.8509
  Wrong [EN] 'boundaries canada lands indian lands' → not found | score: 0.8653
  Wrong [EN] 'Age Men Women' → not found | score: 0.8407
  Wrong [EN] 'society and community table volunteering and donat' → not found | score: 0.8398
  Right [EN] 'controlled and illegal' → rank #1 | score: 0.8359
  Wrong [EN] 'boundaries canada lands indian lands' → not found | score: 0.8653
  Wrong [EN] 'airborne-geophysics ygs-import ygs-import-20250711' → not found | score: 0.8824
  Right [

In [ ]:
#Plot 

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Semantic Search Evaluation Results", fontsize=13, fontweight='bold')

# Plot 1 — Hit Rate comparison
langs      = ["English", "French", "Overall"]
hit_rates  = [en_hit_rate, fr_hit_rate, overall]
colors     = ["#2196F3", "#4CAF50","#FF9800"]
bars = axes[0].bar(langs, hit_rates, color=colors, width=0.5)
axes[0].set_title("Hit Rate @5 by Language")
axes[0].set_ylabel("Hit Rate (%)")
axes[0].set_ylim(0, 110)
axes[0].bar_label(bars, fmt='%.1f%%', padding=3)
axes[0].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% target')
axes[0].legend()

# Plot 2 — Score distribution
axes[1].hist(en_results["top_score"], bins=8, alpha=0.6, color="#2196F3", label="English")
axes[1].hist(fr_results["top_score"], bins=8, alpha=0.6, color="#4CAF50", label="French")
axes[1].set_title("Top Score Distribution")
axes[1].set_xlabel("Similarity Score")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig(data_folder + "evaluation_results.png", dpi=150, bbox_inches='tight')
plt.show()


results_dataframe.to_csv(data_folder + "evaluation_results.csv", index=False)
print("done")